# CT.M1 – Data Acquisition
### Notebook 03. ChEMBL Data Retrieval 

**Version 1.0.0 - February, 2026. Monterrey**

**Author:** Flavio F. Contreras-Torres


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/CheminformaticsTutorial/blob/main/01_Data_Acquisition/notebooks/01_pubchem_data_retrieval.ipynb)


---


### Contents

This notebook mirrors the structure of the PubChem notebook, but using **ChEMBL**.

**Key idea:** 
ChEMBL is bioactivity-centered (assays/targets/activities). You’ll typically retrieve:
- Molecules (structures + computed properties)
- Activities (IC50/EC50/Ki, etc.)
- Assays and Targets (biological context)

---

1. Step 0 – Computational environment
2. Step 1 – Defining the search space (natural language → ChEMBL IDs)
3. Step 2 – Retrieving molecule records (SMILES, InChIKey, basic properties)
4. Step 3 – QC and initial inspection
5. Step 4 – Optional: retrieve bioactivities for your molecules
6. Step 5 – Save dataset



### How to work

1. **Open the notebook**: Click on the "Open in Colab" button or use the link provided to open the tutorial in **Google Colab**.
2. **Create your workspace**: Once open, go to **File > Save a copy in Drive**. This is vital! You must work on your own copy to ensure your progress is saved.
    - **Tip**: Look at the top-left corner. If you see "Copy of...", you are ready to go!
3. **Solve the exercises**: Complete the missing parts of the code where indicated. You can run each cell to test your progress (Shift + Enter).
4. When you finish:
    - **Rename** your file following the convention:
        - `YourName_YourID_03_chembl_data_retrieval.ipynb`
    - **Download the file**: Go to File > Download > Download .ipynb.
    - **Upload** the downloaded file to the **CANVAS assignment module**.

**Do not** modify the notebook structure or function signatures unless explicitly stated.


---


## ChEMBL Data Retrieval  

Programmatic access to databases is an efficient and automatable way to retrieve specific, relevant information for scientific research tasks. Among the available repositories, **PubChem** is one of the most widely consulted by the scientific community. It provides access to tens of millions of data points regarding compounds, substances, and bioassays for protein bioactivity, which serve as key molecular targets.

**ChEMBL** is a public chemical information resource maintained by the NIH, providing access to molecular structures, identifiers, and associated chemical properties.

In this notebook, we will **search PubChem**, **retrieve molecular properties programmatically**, and **construct a structured dataset** suitable for downstream curation and analysis.

---

### Step 0: The Computational Environment

We will use plain `requests` to keep the workflow transparent (no hidden client abstractions).

In [ ]:
import os
import time
import requests
import pandas as pd

print("Environment ready.")
print("requests version:", requests.__version__)


### Step 1: Defining the Chemical Search Space

In **ChEMBL**, a text query is commonly translated into **molecule_chembl_id** via the `molecule/search` endpoint.

Important: ChEMBL responses are **paginated** using `limit` and `offset`.

In [ ]:
# Step 1

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"
OUT_DIR = os.path.join("..", "data_raw")
os.makedirs(OUT_DIR, exist_ok=True)

# Search configuration
QUERY = "flavonoid"      # Try: "aspirin", "capsaicin", "cannabinoid", "kinase inhibitor"
LIMIT = 50               # items per page (keep small for tutorials)
MAX_RESULTS = 200        # safety cap to avoid pulling too much in a tutorial


### Step 2: From Natural Language to ChEMBL IDs

We will:
1) call `/molecule/search.json?q=...`
2) paginate with `limit` + `offset`
3) extract the `molecule_chembl_id` list

In [ ]:
# Step 2

from urllib.parse import quote

def chembl_get(endpoint: str, params: dict | None = None, timeout: int = 30) -> dict:
    """Small helper for GET requests to ChEMBL."""
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

def search_molecules(query: str, limit: int = 20, max_results: int = 200) -> list[str]:
    """Search molecules by text query and return molecule_chembl_id list."""
    q = query  # requests will URL-encode params for us
    offset = 0
    ids: list[str] = []

    while True:
        data = chembl_get("molecule/search.json", params={"q": q, "limit": limit, "offset": offset})
        molecules = data.get("molecules", [])  # search endpoint uses 'molecules'
        if offset == 0:
            print("Example endpoint:", f"{BASE_URL}/molecule/search.json?q={quote(q)}&limit={limit}&offset={offset}")

        for m in molecules:
            mid = m.get("molecule_chembl_id")
            if mid:
                ids.append(mid)

        if len(molecules) < limit:
            break  # last page
        if len(ids) >= max_results:
            ids = ids[:max_results]
            break

        offset += limit
        time.sleep(0.1)

    # de-duplicate while preserving order
    seen = set()
    out = []
    for x in ids:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

molecule_ids = search_molecules(QUERY, limit=LIMIT, max_results=MAX_RESULTS)
print("Total molecule_chembl_id retrieved:", len(molecule_ids))
print("First 10:", molecule_ids[:10])


### Step 3: Retrieve Molecule Records (Structures + Properties)

There are two common ways:
- **One-by-one**: `/molecule/CHEMBL25.json`
- **Bulk filter**: `/molecule.json?molecule_chembl_id__in=CHEMBL25,CHEMBL941,...`

For teaching purposes, we’ll do safe batching with the `__in` filter.

In [ ]:
# Step 3

def fetch_molecules_bulk(molecule_ids: list[str], batch_size: int = 25) -> pd.DataFrame:
    """Fetch molecule records in batches using molecule_chembl_id__in filter."""
    if not molecule_ids:
        return pd.DataFrame()

    rows = []
    for i in range(0, len(molecule_ids), batch_size):
        batch = molecule_ids[i:i+batch_size]
        params = {"molecule_chembl_id__in": ",".join(batch), "limit": batch_size, "offset": 0}
        data = chembl_get("molecule.json", params=params)

        # list endpoints wrap objects in a resource-specific key:
        # for molecule it's 'molecules'
        mols = data.get("molecules", [])
        rows.extend(mols)
        if i == 0:
            print("Example endpoint:", f"{BASE_URL}/molecule.json?molecule_chembl_id__in={batch[0]},{batch[1]}...")

        time.sleep(0.1)

    df = pd.json_normalize(rows)

    # Keep a clean “starter” subset of columns (you can expand later)
    keep = [
        "molecule_chembl_id",
        "pref_name",
        "molecule_structures.canonical_smiles",
        "molecule_structures.standard_inchi_key",
        "molecule_properties.full_mwt",
        "molecule_properties.alogp",
        "molecule_properties.tpsa",
        "max_phase",
        "molecule_type",
    ]
    keep = [c for c in keep if c in df.columns]
    return df[keep].rename(columns={
        "molecule_structures.canonical_smiles": "SMILES",
        "molecule_structures.standard_inchi_key": "InChIKey",
        "molecule_properties.full_mwt": "MolecularWeight",
        "molecule_properties.alogp": "AlogP",
        "molecule_properties.tpsa": "TPSA",
    })

df = fetch_molecules_bulk(molecule_ids, batch_size=25)
df.head()


### Step 4: Initial Data Inspection & Quality Control

Typical QC checks:
- missing SMILES / InChIKey
- duplicates by SMILES
- basic distribution sanity (MW, TPSA, logP)

In [ ]:
# Step 4

print("Dataset shape:", df.shape)
print("Missing SMILES:", df["SMILES"].isna().sum() if "SMILES" in df else "N/A")
print("Missing InChIKey:", df["InChIKey"].isna().sum() if "InChIKey" in df else "N/A")

if "SMILES" in df:
    print("Duplicate SMILES:", df.duplicated(subset=["SMILES"]).sum())

df.describe(include="all").T.head(20)


### Step 4b (Optional): Retrieve Bioactivities for Your Molecules

ChEMBL’s superpower is **activity data**.

Example: pull activities for a given molecule (IC50/Ki/etc.) using:
`/activity.json?molecule_chembl_id=CHEMBL25`

Below we fetch a limited number of activity rows per molecule to keep the tutorial lightweight.

In [ ]:
# Step 4b (Optional)

def fetch_activities_for_molecule(molecule_chembl_id: str, limit: int = 100) -> pd.DataFrame:
    data = chembl_get("activity.json", params={"molecule_chembl_id": molecule_chembl_id, "limit": limit, "offset": 0})
    acts = data.get("activities", [])
    if not acts:
        return pd.DataFrame()

    df = pd.json_normalize(acts)
    keep = [
        "molecule_chembl_id",
        "target_chembl_id",
        "assay_chembl_id",
        "standard_type",
        "standard_relation",
        "standard_value",
        "standard_units",
        "pchembl_value",
        "standard_flag",
    ]
    keep = [c for c in keep if c in df.columns]
    return df[keep]

# Example (comment out if you want to skip activity retrieval)
example_id = molecule_ids[0] if molecule_ids else None
if example_id:
    act_df = fetch_activities_for_molecule(example_id, limit=100)
    print("Example molecule:", example_id)
    print("Activities retrieved:", act_df.shape)
    act_df.head()


### Step 5: Save Dataset

We save the molecule table as CSV (and optionally activities as a second CSV).

In [ ]:
# Step 5

if df.empty:
    print("[!] Warning: DataFrame is empty. Saving empty file.")

filename = f"chembl_raw_{QUERY}.csv".replace(" ", "_")
out_path = os.path.join(OUT_DIR, filename)
df.to_csv(out_path, index=False)

print(f"File successfully saved to: {out_path}")
print(f"Dataset shape: {df.shape}")

# Optional: save activities if they exist
if "act_df" in globals() and isinstance(act_df, pd.DataFrame) and not act_df.empty:
    act_path = os.path.join(OUT_DIR, f"chembl_activities_{QUERY}_{example_id}.csv".replace(" ", "_"))
    act_df.to_csv(act_path, index=False)
    print(f"Activities saved to: {act_path}")


### Summary and Next Steps

- You now have a **molecule-level** dataset from ChEMBL.
- Next, we can build a **target-centric** dataset (e.g., PPARγ), aggregate activities, and create a clean ML-ready table.

### Exercises
1. Change `QUERY` to specific molecules: `aspirin`, `capsaicin`, `cannabidiol`.
2. Compare missing values in `AlogP` or `TPSA` across queries.
3. For a molecule with many activities, filter by `standard_type == 'IC50'` and explore `pchembl_value`.
